# Full Training Pipeline - Hand Pose Estimation Model (CUDA Optimized)

Notebook này được tối ưu để chạy full training trên VM GPU NVIDIA (RTX 5060 Ti 16GB), đồng thời vẫn hỗ trợ chạy local trên macOS để kiểm tra nhanh.

**Mục tiêu tối ưu hiệu năng:**
- Cài đúng TensorFlow GPU stack theo OS
- Bật Mixed Precision cho Tensor Cores
- Bật XLA JIT để tăng tốc kernel
- Dùng train step chạy trong TensorFlow graph
- Tối ưu input pipeline cho throughput cao

**Cấu hình mặc định khuyến nghị cho RTX 5060 Ti 16GB:**
- Backbone: MobileNetV3Small
- Batch Size: 32 (có thể tăng 40-48 nếu còn VRAM)
- Learning Rate: 1e-3 + Exponential Decay
- Epochs: 50+
- Checkpoint: best + periodic

## 1. Install Required Libraries

In [ ]:
import platform
import subprocess
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())

is_linux = platform.system().lower() == "linux"
is_macos = platform.system().lower() == "darwin"

# Always upgrade core packaging tools first
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])

if is_linux:
    # Linux VM with NVIDIA GPU: use TensorFlow CUDA wheel
    linux_packages = [
        "tensorflow[and-cuda]==2.16.1",
        "numpy>=1.24.0,<2.0.0",
        "scipy>=1.10.0",
        "pillow>=9.5.0",
        "opencv-python-headless>=4.9.0.80",
        "tensorboard>=2.16.0",
    ]
    print("\nInstalling Linux CUDA training stack...")
    for pkg in linux_packages:
        print(f"  - {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", pkg])

elif is_macos:
    # Local MacBook testing (Apple Silicon / Metal)
    mac_packages = [
        "tensorflow-macos>=2.13.0",
        "tensorflow-metal>=1.1.0",
        "numpy>=1.24.0,<2.0.0",
        "scipy>=1.10.0",
        "pillow>=9.5.0",
        "opencv-python>=4.9.0.80",
        "tensorboard>=2.13.0",
    ]
    print("\nInstalling macOS training stack...")
    for pkg in mac_packages:
        print(f"  - {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", pkg])

else:
    # Fallback environment
    generic_packages = [
        "tensorflow==2.16.1",
        "numpy>=1.24.0,<2.0.0",
        "scipy>=1.10.0",
        "pillow>=9.5.0",
        "opencv-python-headless>=4.9.0.80",
        "tensorboard>=2.16.0",
    ]
    print("\nInstalling generic training stack...")
    for pkg in generic_packages:
        print(f"  - {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", pkg])

print("\nInstallation complete. Restart kernel before continuing if this is first setup.")

## 1.1 Configuration Parameters (Optional - Modify as needed)

In [ ]:
# ========== TRAINING CONFIGURATION FOR RTX 5060 Ti (16GB) ==========

# Dataset
BATCH_SIZE = 32                # Recommended start for 16GB VRAM
IMAGE_SIZE = 224

# Model
BACKBONE_MODEL = "mobilenetv3small"  # "mobilenetv3small" or "resnet50"
TRAINABLE_BACKBONE = True

# Performance knobs
USE_MIXED_PRECISION = True     # Great speedup on NVIDIA RTX
ENABLE_XLA = True              # JIT compile kernels
SET_MEMORY_GROWTH = True       # Stable memory behavior

# Optimizer / LR schedule
INITIAL_LEARNING_RATE = 1e-3
DECAY_STEPS = 1000
DECAY_RATE = 0.96

# Training
NUM_TRAINING_EPOCHS = 50
LOG_EVERY_N_STEPS = 20

# Checkpoint
SAVE_BEST_ONLY = True
SAVE_PERIODIC = True
PERIODIC_SAVE_INTERVAL = 10

print("Configuration loaded:")
print(f"  BATCH_SIZE={BATCH_SIZE}")
print(f"  BACKBONE_MODEL={BACKBONE_MODEL}")
print(f"  USE_MIXED_PRECISION={USE_MIXED_PRECISION}")
print(f"  ENABLE_XLA={ENABLE_XLA}")
print(f"  NUM_TRAINING_EPOCHS={NUM_TRAINING_EPOCHS}")

## 2. Import Dependencies and Setup Paths

In [ ]:
from __future__ import annotations

import os
import platform
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow.keras import mixed_precision

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

REPO_ROOT = Path(".").resolve().parent
print(f"Repository root: {REPO_ROOT}")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from data.dataset import build_dataset
from models.pose_net import create_pose_net, HEATMAP_SIZE
from training.losses import pose_loss

print("Imports successful")
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("CUDA build:", tf.test.is_built_with_cuda())
print("OS:", platform.platform())
print("nvidia-smi path:", shutil.which("nvidia-smi"))

# Configure memory growth early (before first heavy GPU op)
gpus = tf.config.list_physical_devices("GPU")
set_memory_growth_flag = globals().get("SET_MEMORY_GROWTH", True)
if gpus and set_memory_growth_flag:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("Enabled memory growth for GPUs")

if shutil.which("nvidia-smi"):
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
            text=True,
        ).strip()
        print("GPU(s):")
        print(out)
    except Exception as exc:
        print("Could not query nvidia-smi:", exc)

In [ ]:
print("## 2.1 CUDA / GPU Sanity Check")

gpus = tf.config.list_physical_devices("GPU")
print("Visible GPUs:", gpus)

if not gpus:
    raise RuntimeError(
        "Không thấy GPU trong TensorFlow. Kiểm tra lại driver NVIDIA + cài tensorflow[and-cuda]."
    )

build_info = tf.sysconfig.get_build_info()
print("CUDA version (TF build):", build_info.get("cuda_version", "unknown"))
print("cuDNN version (TF build):", build_info.get("cudnn_version", "unknown"))

# Lightweight warmup to verify kernels can launch
with tf.device("/GPU:0"):
    x = tf.random.normal((1024, 1024))
    y = tf.random.normal((1024, 1024))
    z = tf.matmul(x, y)
print("GPU warmup OK. shape:", z.shape)

print("CUDA sanity check passed")

## 3. Load and Prepare Dataset

In [ ]:
print("Building full training dataset...")
print(f"  Batch size: {BATCH_SIZE}")
print("  Shuffle: True")
print("  Augmentation: True")

train_dataset = build_dataset(
    batch_size=BATCH_SIZE,
    shuffle=True,
    training=True,
)

# Dataset options for throughput
options = tf.data.Options()
options.experimental_deterministic = False
train_dataset = train_dataset.with_options(options)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

sample_batch = next(iter(train_dataset.take(1)))
print("\nDataset sample:")
print(f"  image: {sample_batch['image'].shape} {sample_batch['image'].dtype}")
print(f"  keypoints: {sample_batch['keypoints'].shape} {sample_batch['keypoints'].dtype}")
print(f"  K: {sample_batch['K'].shape} {sample_batch['K'].dtype}")

images = sample_batch["image"].numpy()
print(f"  image range: [{images.min():.4f}, {images.max():.4f}]")

card = tf.data.experimental.cardinality(train_dataset)
if card.numpy() > 0:
    print(f"  steps/epoch (estimated): {int(card.numpy())}")
else:
    print("  steps/epoch: unknown (streaming cardinality)")

## 4. Build and Configure Model

In [ ]:
physical_gpus = tf.config.list_physical_devices("GPU")
print("Detected GPUs:", physical_gpus)

if physical_gpus:
    print(f"Using GPU: {physical_gpus[0].name}")
else:
    print("No GPU detected. Training will run on CPU (slow).")

if USE_MIXED_PRECISION and physical_gpus:
    mixed_precision.set_global_policy("mixed_float16")
else:
    mixed_precision.set_global_policy("float32")
print("Global mixed precision policy:", mixed_precision.global_policy())

tf.config.optimizer.set_jit(ENABLE_XLA)
print("XLA enabled:", ENABLE_XLA)

print("\nCreating model...")
print(f"  Backbone: {BACKBONE_MODEL}")
print(f"  Trainable backbone: {TRAINABLE_BACKBONE}")

model = create_pose_net(backbone_name=BACKBONE_MODEL, trainable_backbone=TRAINABLE_BACKBONE)
model.summary()

trainable_params = int(np.sum([np.prod(v.shape) for v in model.trainable_variables]))
print(f"Trainable params: {trainable_params:,}")

## 5. Helper Functions for Data Preparation

In [ ]:
@tf.function(reduce_retracing=True)
def prepare_batch_uvz_tf(keypoints_xyz: tf.Tensor, K: tf.Tensor, image_size: int = IMAGE_SIZE) -> tf.Tensor:
    """Vectorized xyz -> uvz conversion in TF graph for high throughput."""
    eps = tf.constant(1e-6, dtype=tf.float32)
    xyz = tf.cast(keypoints_xyz, tf.float32)  # (B, 21, 3)
    K = tf.cast(K, tf.float32)                # (B, 3, 3)

    x = xyz[..., 0]
    y = xyz[..., 1]
    z = xyz[..., 2]

    fx = K[:, None, 0, 0]
    fy = K[:, None, 1, 1]
    cx = K[:, None, 0, 2]
    cy = K[:, None, 1, 2]

    u = fx * (x / (z + eps)) + cx
    v = fy * (y / (z + eps)) + cy

    # Relative depth wrt root joint index 0
    z_rel = z - z[:, :1]

    scale = tf.cast(HEATMAP_SIZE, tf.float32) / tf.cast(image_size, tf.float32)
    u = u * scale
    v = v * scale

    return tf.stack([u, v, z_rel], axis=-1)  # (B, 21, 3)


print("TF graph helper ready: prepare_batch_uvz_tf")

## 6. Setup Training Components (Optimizer, LR Scheduler, Logging)

In [ ]:
INITIAL_LR = INITIAL_LEARNING_RATE
NUM_EPOCHS = NUM_TRAINING_EPOCHS

print("Setting up training components...")

# Exponential LR decay
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=INITIAL_LR,
    decay_steps=DECAY_STEPS,
    decay_rate=DECAY_RATE,
    staircase=True,
)
optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

# Checkpoint + logs
ckpt_dir = REPO_ROOT / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)

log_dir = REPO_ROOT / "logs" / "pose_full" / datetime.now().strftime("%Y%m%d-%H%M%S")
log_dir.mkdir(parents=True, exist_ok=True)
summary_writer = tf.summary.create_file_writer(str(log_dir))

best_loss = float("inf")
global_step = 0

# Compiled train step for better GPU utilization
@tf.function(jit_compile=ENABLE_XLA, reduce_retracing=True)
def train_step(images: tf.Tensor, keypoints: tf.Tensor, K: tf.Tensor) -> tf.Tensor:
    gt_uvz = prepare_batch_uvz_tf(keypoints, K, image_size=IMAGE_SIZE)
    with tf.GradientTape() as tape:
        outputs = model(images, training=True)
        loss = pose_loss(outputs["heatmap"], outputs["depthmap"], gt_uvz)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

print("Optimizer, train_step, checkpoints, and TensorBoard writer are ready")
print(f"Log dir: {log_dir}")
print(f"Checkpoint dir: {ckpt_dir}")

## 7. Execute Full Training Loop

In [ ]:
print("=" * 80)
print("FULL TRAINING START")
print("=" * 80)
print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Mixed precision: {mixed_precision.global_policy()}")
print(f"XLA: {ENABLE_XLA}")
print(f"Logs: {log_dir}")
print(f"Checkpoints: {ckpt_dir}")
print("=" * 80)

try:
    for epoch in range(NUM_EPOCHS):
        epoch_losses = []

        for batch_idx, batch in enumerate(train_dataset):
            loss = train_step(batch["image"], batch["keypoints"], batch["K"])
            loss_value = float(loss.numpy())
            epoch_losses.append(loss_value)
            global_step += 1

            if (batch_idx + 1) % LOG_EVERY_N_STEPS == 0:
                with summary_writer.as_default():
                    tf.summary.scalar("Loss/train_step", loss_value, step=global_step)

        avg_loss = float(np.mean(epoch_losses))
        current_lr = float(lr_schedule(optimizer.iterations).numpy())

        with summary_writer.as_default():
            tf.summary.scalar("Loss/train_epoch", avg_loss, step=epoch)
            tf.summary.scalar("Learning_Rate", current_lr, step=epoch)

        print(
            f"Epoch {epoch + 1:03d}/{NUM_EPOCHS} | "
            f"loss={avg_loss:.6f} | lr={current_lr:.3e}"
        )

        if avg_loss < best_loss:
            best_loss = avg_loss
            best_path = ckpt_dir / "pose_best.weights.h5"
            model.save_weights(str(best_path))
            print(f"  Saved best checkpoint: {best_path.name} (loss={best_loss:.6f})")

        if SAVE_PERIODIC and (epoch + 1) % PERIODIC_SAVE_INTERVAL == 0:
            periodic_path = ckpt_dir / f"pose_epoch_{epoch + 1:03d}.weights.h5"
            model.save_weights(str(periodic_path))
            print(f"  Saved periodic checkpoint: {periodic_path.name}")

except KeyboardInterrupt:
    print("Training interrupted by user")
except Exception as e:
    import traceback

    print(f"Training failed: {e}")
    traceback.print_exc()

print("=" * 80)
print("TRAINING COMPLETE")
print("=" * 80)
print(f"Best loss: {best_loss:.6f}")
print(f"Best checkpoint: {ckpt_dir / 'pose_best.weights.h5'}")
print(f"TensorBoard logs: {log_dir}")

## 8. Monitor Training with TensorBoard

In [ ]:
print("=" * 80)
print("TENSORBOARD MONITORING")
print("=" * 80)
print("Run on VM terminal:")
print(f"tensorboard --logdir {REPO_ROOT / 'logs'} --port 6006 --bind_all")
print()
print("If access from local machine via SSH tunnel:")
print("ssh -L 6006:localhost:6006 <user>@<vm_ip>")
print("Then open: http://localhost:6006")
print()
print("Tracked metrics:")
print("- Loss/train_step")
print("- Loss/train_epoch")
print("- Learning_Rate")
print("=" * 80)

## 9. Post-Training Utilities

In [ ]:
print("=" * 80)
print("CHECKPOINTS")
print("=" * 80)

if ckpt_dir.exists():
    checkpoints = sorted(ckpt_dir.glob("*.h5"))
    if checkpoints:
        print(f"Found {len(checkpoints)} checkpoint(s):")
        for ckpt in checkpoints:
            size_mb = ckpt.stat().st_size / (1024 ** 2)
            print(f"  - {ckpt.name:40s} {size_mb:8.2f} MB")
    else:
        print("No checkpoints found yet")
else:
    print(f"Checkpoint directory not found: {ckpt_dir}")

print("\n" + "=" * 80)
print("VM PERFORMANCE TIPS FOR RTX 5060 Ti 16GB")
print("=" * 80)
print("1) Start with BATCH_SIZE=32 on MobileNetV3Small")
print("2) If GPU VRAM usage < 13GB, increase BATCH_SIZE to 40 or 48")
print("3) If OOM occurs, reduce BATCH_SIZE to 24")
print("4) Keep USE_MIXED_PRECISION=True and ENABLE_XLA=True")
print("5) Use backbone mobilenetv3small for max throughput, resnet50 for higher capacity")
print("6) Monitor with: watch -n 1 nvidia-smi")
print("=" * 80)